In [3]:
import os
import sys
import pandas as pd

# 将项目根目录加入 Python 路径
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        print(f"✅ 已添加项目根目录: {project_root}")

from config import get_driver
from src.retriever import find_matching_phages, find_similar_cases

print("✅ 模块导入完成")

✅ 模块导入完成


In [4]:
# ========== 导入合作方“黄金配型”知识库（整合版：节点+关系一步到位） ==========
from src.data_loader import get_driver

validated_rules = [
    {
        "rule_id": "RULE_CRAB_KL2",
        "pathogen_species": "Acinetobacter baumannii",
        "strain_type": "KL2",
        "phage_name": "ΦK2-v3",
        "treatment": "ΦK2-v3 单用",
        "outcome": "第14天微生物清除，第6个月未复发",
        "evidence_from": "肖易倍团队第N次配型"
    },
    {
        "rule_id": "RULE_CRKP_KL47",
        "pathogen_species": "Klebsiella pneumoniae",
        "strain_type": "KL47",
        "phage_name": "ΦK47-w7",
        "treatment": "ΦK47-w7 + 碳青霉烯类",
        "outcome": "第3个月未复发",
        "evidence_from": "肖易倍团队第N次配型"
    },
    {
        "rule_id": "RULE_ECOLI_O25",
        "pathogen_species": "Escherichia coli",
        "strain_type": "O25",
        "phage_name": "CP-p-EC-23086",
        "treatment": "膀胱灌注（局部递送）",
        "outcome": "48小时内细菌计数断崖式下降",
        "evidence_from": "临床验证（结合CASE-001及既往数据）"
    }
]

with get_driver() as driver:
    with driver.session() as session:
        for rule in validated_rules:
            result = session.run("""
                // 1. 创建/更新知识规则节点
                MERGE (r:KnowledgeRule {rule_id: $rule_id})
                SET r.strain_type = $strain_type,
                    r.treatment = $treatment,
                    r.outcome = $outcome,
                    r.evidence_from = $evidence_from
                WITH r

                // 2. 关联到病原菌（若不存在则创建）
                MERGE (p:Pathogen {species: $pathogen_species})
                ON CREATE SET p.resistance_mechanism = 'Unknown'
                MERGE (p)-[:HAS_VALIDATED_RULE]->(r)
                WITH r

                // 3. 关联到噬菌体（若不存在则创建）
                MERGE (ph:Phage {name: $phage_name})
                ON CREATE SET ph.family = 'Unknown'
                MERGE (r)-[:RECOMMENDS_PHAGE]->(ph)

                RETURN r.rule_id AS id
            """,
            rule_id=rule["rule_id"],
            pathogen_species=rule["pathogen_species"],
            strain_type=rule["strain_type"],
            phage_name=rule["phage_name"],
            treatment=rule["treatment"],
            outcome=rule["outcome"],
            evidence_from=rule["evidence_from"]
            )

            record = result.single()
            if record:
                print(f"✅ 导入规则: {record['id']} (关联噬菌体: {rule['phage_name']})")
            else:
                print(f"❌ 导入规则 {rule['rule_id']} 失败")

print("\n🎉 黄金配型知识库导入完成！（规则 + 病原菌关联 + 噬菌体关联 已全部建立）")

✅ 导入规则: RULE_CRAB_KL2 (关联噬菌体: ΦK2-v3)
✅ 导入规则: RULE_CRKP_KL47 (关联噬菌体: ΦK47-w7)
✅ 导入规则: RULE_ECOLI_O25 (关联噬菌体: CP-p-EC-23086)

🎉 黄金配型知识库导入完成！（规则 + 病原菌关联 + 噬菌体关联 已全部建立）


In [5]:
# ============================================================
# 创建测试病例（CRAB KL2 + CRKP KL47）
# 依赖：黄金规则（RULE_CRAB_KL2 / RULE_CRKP_KL47）已存在
# ============================================================
from src.data_loader import get_driver

# -------- 定义两条 Cypher 语句（分别创建两个病例） --------
cypher_crab = """
MERGE (p:Pathogen {species: "Acinetobacter baumannii"})
SET p.resistance_mechanism = "Carbapenem-resistant",
    p.strain_type = "KL2",
    p.verification_status = "MICROBIOLOGY_LAB_VERIFIED"
WITH p
CREATE (c:ClinicalCase {
    case_id: "CASE-999",
    infection_type: "Pneumonia",
    infection_site: "Lung",
    specimen_type: "Sputum",
    patient_age_group: "65-75",
    comorbidities: ["COPD", "Diabetes"],
    prior_antibiotics: ["Meropenem", "Colistin"],
    phage_treatment: null,
    clinical_outcome: null,
    microbiological_outcome: null,
    curated_by: "FDE-TEST",
    curation_date: "2026-07-13"
})
WITH c, p
MERGE (c)-[:INVOLVES_PATHOGEN]->(p)
MATCH (r:KnowledgeRule {rule_id: "RULE_CRAB_KL2"})
MERGE (p)-[:HAS_VALIDATED_RULE]->(r)
RETURN "CASE-999 创建成功" AS result
"""

cypher_kp = """
MERGE (p:Pathogen {species: "Klebsiella pneumoniae"})
SET p.resistance_mechanism = "Carbapenem-resistant",
    p.strain_type = "KL47",
    p.verification_status = "MICROBIOLOGY_LAB_VERIFIED"
WITH p
CREATE (c:ClinicalCase {
    case_id: "CASE-998",
    infection_type: "VAP",
    infection_site: "Lung",
    specimen_type: "BALF",
    patient_age_group: "55-65",
    comorbidities: ["Diabetes", "Immunosuppression"],
    prior_antibiotics: ["Meropenem"],
    phage_treatment: null,
    clinical_outcome: null,
    microbiological_outcome: null,
    curated_by: "FDE-TEST",
    curation_date: "2026-07-13"
})
WITH c, p
MERGE (c)-[:INVOLVES_PATHOGEN]->(p)
MATCH (r:KnowledgeRule {rule_id: "RULE_CRKP_KL47"})
MERGE (p)-[:HAS_VALIDATED_RULE]->(r)
RETURN "CASE-998 创建成功" AS result
"""

# -------- 执行两条 Cypher --------
with get_driver() as driver:
    with driver.session() as session:
        # 执行 CRAB 病例
        result_crab = session.run(cypher_crab)
        print(result_crab.single()['result'])
        
        # 执行 CRKP 病例
        result_kp = session.run(cypher_kp)
        print(result_kp.single()['result'])

print("\n✅ 测试病例创建完成！")

CASE-999 创建成功
CASE-998 创建成功

✅ 测试病例创建完成！


In [7]:
# ========== 基于规则的推荐引擎（不依赖 LLM ）==========
from src.package_builder import rule_based_evidence_package
import json

# 针对 CRAB KL2
result_crab = rule_based_evidence_package(
    species="Acinetobacter baumannii",# Klebsiella pneumoniae
    resistance="Carbapenem-resistant",# Carbapenem-resistant
    infection_type="Pneumonia"# VAP
)
print("CRAB KL2 推荐结果：")
print(json.dumps(result_crab, ensure_ascii=False, indent=2))

CRAB KL2 推荐结果：
{
  "matching_evidence": [
    {
      "phage_name": "ΦK2-v3",
      "family": "黄金规则推荐",
      "infection_result": "Lytic (黄金规则)",
      "infection_probability": 1.0,
      "evidence_level": "GOLDEN_RULE",
      "evidence_ref": [
        "Rule: RULE_CRAB_KL2"
      ],
      "notes": "由经过临床验证的黄金规则推荐。预期结局：第14天微生物清除，第6个月未复发"
    },
    {
      "phage_name": "vB_AbaM_003",
      "family": "Siphoviridae",
      "infection_result": "Lytic",
      "infection_probability": 0.82,
      "evidence_level": "L3",
      "evidence_ref": [
        "CASE-004"
      ],
      "notes": "临床验证有效"
    },
    {
      "phage_name": "vB_AbaM_007",
      "family": "Myoviridae",
      "infection_result": "Lytic",
      "infection_probability": 0.8,
      "evidence_level": "L3",
      "evidence_ref": [
        "CASE-013"
      ],
      "notes": "临床验证有效"
    },
    {
      "phage_name": "vB_AbaM_005",
      "family": "Myoviridae",
      "infection_result": "Lytic",
      "infection_probability": 0.78

In [18]:
# ========== 基于规则的依赖 LLM 的推荐==========
import json
from src.package_builder import build_evidence_package_from_db

# 针对 CRAB KL2
llm_crab = build_evidence_package_from_db(
    species="Acinetobacter baumannii",
    resistance="Carbapenem-resistant",
    infection_type="Pneumonia"
)
print("🤖 CRAB KL2 LLM 推荐结果：")
print(json.dumps(llm_crab, ensure_ascii=False, indent=2))
print(" ========== 内容单独展示========== ")
print(llm_crab.get('explanation'))

🤖 CRAB KL2 LLM 推荐结果：
{
  "matching_evidence": [
    {
      "phage_name": "ΦK2-v3",
      "family": "黄金规则推荐",
      "infection_result": "Lytic (黄金规则)",
      "infection_probability": 1.0,
      "evidence_level": "GOLDEN_RULE",
      "evidence_ref": [
        "Rule: RULE_CRAB_KL2"
      ],
      "notes": "由经过临床验证的黄金规则推荐。预期结局：第14天微生物清除，第6个月未复发"
    },
    {
      "phage_name": "vB_AbaM_003",
      "family": "Siphoviridae",
      "infection_result": "Lytic",
      "infection_probability": 0.82,
      "evidence_level": "L3",
      "evidence_ref": [
        "CASE-004"
      ],
      "notes": "临床验证有效"
    },
    {
      "phage_name": "vB_AbaM_007",
      "family": "Myoviridae",
      "infection_result": "Lytic",
      "infection_probability": 0.8,
      "evidence_level": "L3",
      "evidence_ref": [
        "CASE-013"
      ],
      "notes": "临床验证有效"
    },
    {
      "phage_name": "vB_AbaM_005",
      "family": "Myoviridae",
      "infection_result": "Lytic",
      "infection_probability"

In [11]:
# ============================================================
# 调用 DeepSeek → 输出完整的三部分 Evidence Package
# 动态高亮：自动提取 evidence_ref 中的 CASE-XXX 来源
# ============================================================
import json
from src.package_builder import build_evidence_package_from_db

# ---------- 调用 LLM 生成 Evidence Package ----------
print("=" * 70)
print("🤖 正在调用 DeepSeek 生成 Evidence Package...")
print("=" * 70)

# 针对 CRKP KL47（用公开数据验证过的菌株）
result = build_evidence_package_from_db(
    species="Klebsiella pneumoniae",
    resistance="Carbapenem-resistant",
    infection_type="VAP"
)

# ---------- 输出完整的三部分 ----------
print("\n" + "=" * 70)
print("📦 Evidence Package（完整输出）")
print("=" * 70)
print(json.dumps(result, ensure_ascii=False, indent=2))

# ---------- 高亮 Clinical Evidence（动态提取 CASE 引用） ----------
print("\n" + "=" * 70)
print("🔬 Clinical Evidence（高亮含噬菌体治疗的病例）")
print("=" * 70)

clinical = result.get('clinical_evidence', [])
for item in clinical:
    case_id = item.get('case_id', '未知')
    treatment = item.get('phage_treatment')
    outcome = item.get('clinical_outcome', '未知')
    
    if treatment and str(treatment).strip() and str(treatment).strip() != 'nan':
        print(f"⭐ {case_id}: {treatment} → {outcome} ✅ (已使用噬菌体治疗)")
    else:
        print(f"   {case_id}: 未使用噬菌体 → {outcome}")

# ---------- 高亮 Matching Evidence（动态提取证据来源） ----------
print("\n" + "=" * 70)
print("🧬 Matching Evidence（动态提取证据来源）")
print("=" * 70)

matching = result.get('matching_evidence', [])

for item in matching:
    phage = item.get('phage_name', '未知')
    level = item.get('evidence_level', '')
    refs = item.get('evidence_ref', [])
    notes = item.get('notes', '')
    
    # 动态提取所有 CASE- 开头的引用
    case_refs = [ref for ref in refs if isinstance(ref, str) and ref.startswith('CASE-')]
    
    # 动态提取所有 PMID 引用
    pmid_refs = [ref for ref in refs if isinstance(ref, str) and ref.startswith('PMID:')]
    
    # 构建来源描述（完全动态）
    source_parts = []
    
    if case_refs:
        source_parts.append(f"来源: {', '.join(case_refs)}")
    
    if notes:
        if '验证' in notes or '黄金规则' in notes or '临床' in notes:
            source_parts.append(notes)
        else:
            source_parts.append(f"备注: {notes}")
    
    if pmid_refs and not case_refs:
        source_parts.append(f"文献来源: {', '.join(pmid_refs)}")
    
    # 根据证据等级决定显示样式
    if case_refs:
        print(f"⭐ {phage} (L{level}) ← {' | '.join(source_parts)} ✅")
    elif level == 'GOLDEN_RULE':
        print(f"⭐ {phage} (黄金规则) ← 经过临床验证的配型知识 ✅")
    elif level == 'L3':
        print(f"   {phage} (L{level}) ← 临床验证 (来源: {', '.join(case_refs) if case_refs else '数据库'})")
    elif source_parts:
        print(f"   {phage} (L{level}) ← {' | '.join(source_parts)}")
    else:
        print(f"   {phage} (L{level}) ← 体外验证")

# ---------- 统计摘要 ----------
print("\n" + "=" * 70)
print("📊 证据统计摘要")
print("=" * 70)

level_counts = {}
for item in matching:
    level = item.get('evidence_level', '未知')
    level_counts[level] = level_counts.get(level, 0) + 1

print(f"匹配噬菌体总数: {len(matching)}")
print(f"证据等级分布:")
for level, count in sorted(level_counts.items()):
    print(f"  - {level}: {count} 条")

# 动态统计引用来源
all_refs = []
for item in matching:
    all_refs.extend(item.get('evidence_ref', []))
case_refs_all = [ref for ref in all_refs if isinstance(ref, str) and ref.startswith('CASE-')]
pmid_refs_all = [ref for ref in all_refs if isinstance(ref, str) and ref.startswith('PMID:')]

print(f"\n引用的临床病例: {set(case_refs_all) if case_refs_all else '无'}")
print(f"引用的文献: {set(pmid_refs_all) if pmid_refs_all else '无'}")

# 检查是否包含黄金规则
has_golden = any(item.get('evidence_level') == 'GOLDEN_RULE' for item in matching)
if has_golden:
    print("\n✅ 成功引用黄金规则")
else:
    print("\n⚠️ 未引用黄金规则（可能该菌种暂无关联规则）")

🤖 正在调用 DeepSeek 生成 Evidence Package...

📦 Evidence Package（完整输出）
{
  "matching_evidence": [
    {
      "phage_name": "ΦK47-w7",
      "family": "黄金规则推荐",
      "infection_result": "Lytic (黄金规则)",
      "infection_probability": 1.0,
      "evidence_level": "GOLDEN_RULE",
      "evidence_ref": [
        "Rule: RULE_CRKP_KL47"
      ],
      "notes": "由经过临床验证的黄金规则推荐。预期结局：第3个月未复发"
    },
    {
      "phage_name": "vB_Kpn_001",
      "family": "Myoviridae",
      "infection_result": "Lytic",
      "infection_probability": 0.91,
      "evidence_level": "L3",
      "evidence_ref": [
        "CASE-005"
      ],
      "notes": "临床验证有效"
    },
    {
      "phage_name": "vB_Kpn_003",
      "family": "Siphoviridae",
      "infection_result": "Lytic",
      "infection_probability": 0.85,
      "evidence_level": "L2",
      "evidence_ref": [],
      "notes": null
    },
    {
      "phage_name": "vB_Kpn_004",
      "family": "Myoviridae",
      "infection_result": "Lytic",
      "infection_probabil

In [19]:
# ============================================================
# 测试跨病例复用分析（FR-7）
# ============================================================
from src.retriever import analyze_cross_case_reuse_simple
import json

print("=" * 70)
print("🔍 测试跨病例复用分析: CASE-001 → CASE-003")
print("=" * 70)

result = analyze_cross_case_reuse_simple("CASE-001", "CASE-003")
print(json.dumps(result, ensure_ascii=False, indent=2))

🔍 测试跨病例复用分析: CASE-001 → CASE-003
{
  "case_a": {
    "id": "CASE-001",
    "species": "Escherichia coli",
    "infection_type": "UTI",
    "phages_used": [
      "CP-p-BC-23062",
      "CP-p-BC-23086"
    ],
    "outcome": "Clinical improvement at Day 7"
  },
  "case_b": {
    "id": "CASE-003",
    "species": "Escherichia coli",
    "infection_type": "UTI",
    "phages_used": [
      "CP-p-BC-23086",
      "CP-p-BC-23062"
    ],
    "outcome": "Improved"
  },
  "reuse_type": "direct_reuse",
  "reused_phages": [
    {
      "name": "CP-p-BC-23062",
      "evidence_level": "L3",
      "probability": 0.96
    },
    {
      "name": "CP-p-BC-23086",
      "evidence_level": "L3",
      "probability": 0.98
    }
  ],
  "reuse_count": 2,
  "is_case_a_similar_to_b": true,
  "is_reuse_valid": true,
  "explanation": "病例 B 直接使用了病例 A 使用过的噬菌体 CP-p-BC-23062, CP-p-BC-23086。"
}


In [14]:
# 1、先找到有 L1/L2 互作的病例（用来验证升级逻辑）
with get_driver() as driver:
    with driver.session() as session:
        result = session.run("""
            MATCH (c:ClinicalCase)-[:TREATED_WITH]->(ph:Phage)-[:HAS_INTERACTION]->(phi:PhageHostInteraction)
            WHERE phi.evidence_level IN ['L1', 'L2']
            RETURN c.case_id, ph.phage_id, ph.name, phi.evidence_level
        """)
        for record in result:
            print(f"{record['c.case_id']} | {record['ph.name']} ({record['ph.phage_id']}) | L{record['phi.evidence_level']}")

CASE-002 | vB_AbaM_001 (PHAGE-013) | LL2


In [15]:
# ============================================================
# 2、测试策展升级：CASE-002 使用 vB_AbaM_001 (当前 L2 → 目标 L3)
# ============================================================
from src.curation import curate_case_by_id
from config import get_driver

with get_driver() as driver:
    summary = curate_case_by_id(
        driver,
        case_id="CASE-002",
        clinical_outcome="Improved",
        microbiological_outcome="Clearance"
    )
    print(summary)

✅ 病例 CASE-002 已更新结局
✅ 升级了 1 条 PhageHostInteraction 至 L3（并追加病例ID）


In [16]:
# ============================================================
# 3、测试策展升级结果查询
# ============================================================
with get_driver() as driver:
    with driver.session() as session:
        result = session.run("""
            MATCH (c:ClinicalCase {case_id: 'CASE-002'})-[r:TREATED_WITH]->(ph:Phage {phage_id: 'PHAGE-013'})
            MATCH (ph)-[:HAS_INTERACTION]->(phi:PhageHostInteraction)
            RETURN ph.name, phi.evidence_level, phi.evidence_ref
        """)
        record = result.single()
        if record:
            print(f"✅ 升级验证：")
            print(f"  噬菌体: {record['ph.name']}")
            print(f"  证据等级: {record['phi.evidence_level']} (应为 L3)")
            print(f"  证据来源: {record['phi.evidence_ref']} (应包含 CASE-002)")

✅ 升级验证：
  噬菌体: vB_AbaM_001
  证据等级: L3 (应为 L3)
  证据来源: ['CASE-002'] (应包含 CASE-002)


In [ ]:
# 交叉验证的演示素材
import pandas as pd
from io import StringIO

data = """phage_name,host_strain,host_sequence_type,host_capsule_type,interaction,source
KP1,KP_001,ST23,KL1,Positive,KlebPhaCol
KP1,KP_002,ST23,KL2,Positive,KlebPhaCol
KP1,KP_003,ST258,KL47,Positive,KlebPhaCol
KP1,KP_004,ST258,KL47,Positive,KlebPhaCol
KP1,KP_005,ST11,KL64,Positive,KlebPhaCol
KP2,KP_001,ST23,KL1,Positive,KlebPhaCol
KP2,KP_003,ST258,KL47,Positive,KlebPhaCol
KP2,KP_006,ST15,KL19,Positive,KlebPhaCol
KP3,KP_002,ST23,KL2,Positive,KlebPhaCol
KP3,KP_004,ST258,KL47,Positive,KlebPhaCol
KP4,KP_001,ST23,KL1,Positive,KlebPhaCol
KP4,KP_007,ST258,KL47,Positive,KlebPhaCol
KP5,KP_003,ST258,KL47,Positive,KlebPhaCol
KP5,KP_008,ST15,KL24,Positive,KlebPhaCol
KP6,KP_001,ST23,KL1,Positive,KlebPhaCol
KP6,KP_009,ST258,KL47,Positive,KlebPhaCol
KP7,KP_002,ST23,KL2,Positive,KlebPhaCol
KP7,KP_010,ST258,KL47,Positive,KlebPhaCol
KP8,KP_003,ST258,KL47,Positive,KlebPhaCol
KP8,KP_011,ST11,KL64,Positive,KlebPhaCol
vB_Kpn_001,KP_012,ST258,KL47,Positive,PHIAF
vB_Kpn_002,KP_013,ST258,KL47,Positive,PHIAF
vB_Kpn_003,KP_014,ST23,KL1,Positive,PHIAF
vB_Kpn_004,KP_015,ST15,KL19,Positive,PHIAF
vB_Kpn_005,KP_016,ST258,KL47,Positive,PHIAF
vB_Kpn_006,KP_017,ST11,KL64,Positive,PHIAF
vB_Kpn_007,KP_018,ST258,KL47,Positive,PHIAF
vB_Kpn_008,KP_019,ST23,KL2,Positive,PHIAF
vB_Kpn_009,KP_020,ST258,KL47,Positive,PHIAF
vB_Kpn_010,KP_021,ST15,KL24,Positive,PHIAF"""

df = pd.read_csv(StringIO(data))
print(f"✅ 加载 {len(df)} 条互作记录")
print(df.head(10))

In [ ]:
driver.close()
print("🔒 数据库连接已关闭")